# Stage 2 evaluation and analysis

Uses the new Stage 1 evaluation contract: intent accuracy and duplicate-aware exact `(slot type, slot text)` micro-F1 from `shared_evaluation.py`. Loads the existing Stage 2 CRF checkpoint; no training or model selection on test data. Run all cells from the repository root.


In [1]:
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
from torchcrf import CRF
from shared_evaluation import intent_accuracy, decode_slots, slot_micro_f1
from collections import Counter
import json, hashlib, platform
from pathlib import Path
import pandas as pd
torch.set_num_threads(4)


cpu


In [2]:
with open("processed.pkl", "rb") as f:
    data = pickle.load(f)

print(data.keys())
print("Train:", len(data["train_data"]))
print("Dev:", len(data["dev_data"]))
print("Test:", len(data["test_data"]))

assert len(data["train_data"]) == 11514
assert len(data["dev_data"]) == 2033
assert len(data["test_data"]) == 2974
word2id = data["word2id"]
tag2id = data["tag2id"]
id2tag = data["id2tag"]
intent2id = data["intent2id"]
id2intent = data["id2intent"]
embedding_matrix = data["embedding_matrix"]

print("Vocabulary size:", len(word2id))
print("BIO tags:", len(tag2id))
print("Intent labels:", len(intent2id))
print("Embedding shape:", embedding_matrix.shape)

assert len(tag2id) == 111
assert len(intent2id) == 60

for label, number in tag2id.items():
    assert id2tag[number] == label

for label, number in intent2id.items():
    assert id2intent[number] == label

dict_keys(['train_data', 'dev_data', 'test_data', 'train_arrays', 'dev_arrays', 'test_arrays', 'word2id', 'tag2id', 'id2tag', 'intent2id', 'id2intent', 'embedding_matrix'])
Train: 11514
Dev: 2033
Test: 2974
Vocabulary size: 5304
BIO tags: 111
Intent labels: 60
Embedding shape: (5304, 300)


## Validate the scorer and data alignment
The same BIO repair and duplicate handling as Stage 1 are used for both models. Padding never contributes to scores. Both validation and test arrays are checked against their original examples.

In [3]:
assert intent_accuracy(['a'], ['a']) == 100
assert intent_accuracy(['a'], ['b']) == 0
assert slot_micro_f1([[('x','a')]], [[('x','a'),('x','a')]]) == {'slot_f1': 200/3, 'tp': 1, 'fp': 1, 'fn': 0}
assert decode_slots(['a','b'], ['I-x','I-x']) == [('x','a b')]
for split, expected in [('dev',2033), ('test',2974)]:
    x, tags, intents, lengths = data[split + '_arrays']
    items = data[split + '_data']
    assert len(x) == len(tags) == len(intents) == len(lengths) == len(items) == expected
    assert x.shape == tags.shape
    for i, item in enumerate(items):
        length = int(lengths[i])
        assert length == len(item['tokens']) > 0
        assert item['tokens'] == item['utt'].split()
        assert x[i,:length].tolist() == [word2id.get(t,1) for t in item['tokens']]
        assert (x[i,:length] != 0).all() and (x[i,length:] == 0).all()
        assert [id2tag[int(t)] for t in tags[i,:length]] == item['tags']
        assert id2intent[int(intents[i])] == item['intent']
        assert decode_slots(item['tokens'], item['tags']) == [tuple(s) for s in item['slots']]
    print(split, expected, 'examples passed alignment checks')


dev 2033 examples passed alignment checks
test 2974 examples passed alignment checks


## Load the trained CRF model
The architecture is copied from Stage 2 training. All weights must match strictly. CRF transitions are learned, not hard BIO constraints.

In [4]:
class JointModel(nn.Module):
    def __init__(self, emb_matrix, n_tags, n_intents, hidden=256, layers=2, dropout=0.4):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(torch.tensor(emb_matrix), freeze=False, padding_idx=0)
        self.emb_drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(emb_matrix.shape[1], hidden, num_layers=layers, batch_first=True,
                            bidirectional=True, dropout=dropout if layers > 1 else 0)
        self.drop = nn.Dropout(dropout)
        self.slot_head = nn.Linear(2 * hidden, n_tags)
        self.intent_head = nn.Linear(4 * hidden, n_intents)
        self.crf = CRF(n_tags, batch_first=True)   # <-- NEW: learns which tag can follow which (e.g. O -> I-time is invalid)

    def forward(self, x, lengths):
        e = self.emb_drop(self.emb(x))
        packed = pack_padded_sequence(e, lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        out = self.drop(out)
        slot_logits = self.slot_head(out)
        mask = (x != 0).unsqueeze(-1)
        mx = out.masked_fill(~mask, -1e9).max(dim=1).values
        mean = (out * mask).sum(1) / mask.sum(1).clamp(min=1)
        intent_logits = self.intent_head(torch.cat([mx, mean], dim=-1))
        return intent_logits, slot_logits


In [5]:
checkpoint_path = 'stage2_best.pt'
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
config = checkpoint['config']
assert config['crf'] and config['word_dropout'] == 0
model = JointModel(embedding_matrix, len(tag2id), len(intent2id), config['hidden'], config['layers'], config['dropout'])
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
model = model.to(device).eval()
total_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert total_parameters == checkpoint['total_params'] <= 20_000_000
assert trainable_parameters == checkpoint['trainable_params']
print('Checkpoint:', checkpoint_path, '| Best epoch:', checkpoint['epoch'])
print('Total / trainable parameters:', total_parameters, '/', trainable_parameters)
print('Config:', config)


Checkpoint: stage2_best.pt | Best epoch: 39
Total / trainable parameters: 4441930 / 4441930
Config: {'epochs': 40, 'batch_size': 32, 'lr': 0.001, 'slot_weight': 1.0, 'hidden': 256, 'layers': 2, 'dropout': 0.5, 'patience': 8, 'clip': 5.0, 'seed': 42, 'word_dropout': 0.0, 'crf': True}


## Infer once per split, then score
Intent uses argmax; slots use CRF Viterbi paths with a padding mask. Predictions retain split row indices for audit. Original span scoring is diagnostic only; the primary score uses the shared Stage 1 functions.

In [6]:
def tags_to_spans(tags):
    """BIO tag strings -> set of (slot_type, start, end_exclusive)."""
    spans, cur_type, start = [], None, None
    for i, t in enumerate(list(tags) + ["O"]):
        if t.startswith("I-") and t[2:] == cur_type:
            continue
        if cur_type is not None:
            spans.append((cur_type, start, i))
        cur_type, start = (t[2:], i) if t != "O" else (None, None)
    return set(spans)


def predict_split(split):
    x, _, _, lengths = [torch.from_numpy(a) for a in data[split + '_arrays']]
    records = []
    with torch.inference_mode():
        for start in range(0,len(x),256):
            lens = lengths[start:start+256]
            inputs = x[start:start+256,:int(lens.max())].to(device)
            intent_logits, emissions = model(inputs,lens)
            intent_ids = intent_logits.argmax(-1).cpu().tolist()
            paths = model.crf.decode(emissions,mask=(inputs != 0))
            for j, path in enumerate(paths):
                i = start+j
                item = data[split + '_data'][i]
                assert len(path) == int(lens[j])
                tags = [id2tag[t] for t in path]
                records.append(dict(index=i, utterance=item['utt'], tokens=item['tokens'],
                    gold_intent=item['intent'], predicted_intent=id2intent[intent_ids[j]],
                    gold_tags=item['tags'], predicted_tags=tags,
                    gold_slots=item['slots'], predicted_slots=decode_slots(item['tokens'],tags)))
    assert len(records) == len(data[split + '_data'])
    return records

def score_records(records):
    return dict(intent_accuracy=intent_accuracy([r['gold_intent'] for r in records], [r['predicted_intent'] for r in records]),
        **slot_micro_f1([r['gold_slots'] for r in records], [r['predicted_slots'] for r in records]))

def span_score(records):
    tp=fp=fn=0
    for r in records:
        gold=tags_to_spans(r['gold_tags']); pred=tags_to_spans(r['predicted_tags'])
        tp+=len(gold & pred); fp+=len(pred-gold); fn+=len(gold-pred)
    return dict(slot_f1=200*tp/(2*tp+fp+fn),tp=tp,fp=fp,fn=fn)

predictions = {split: predict_split(split) for split in ['dev','test']}
scores = {split: score_records(records) for split,records in predictions.items()}
original_span_scores = {split: span_score(records) for split,records in predictions.items()}
assert abs(scores['dev']['intent_accuracy']-checkpoint['val_metrics']['intent_acc']) < 1e-6
assert abs(original_span_scores['dev']['slot_f1']-checkpoint['val_metrics']['slot_f1']) < 1e-6
print('Original validation intent accuracy and span F1 reproduce the checkpoint.')
for split in scores:
    print(split, 'shared Stage 1 scoring:', scores[split])
    print(split, 'original span scoring:', original_span_scores[split])


Original validation intent accuracy and span F1 reproduce the checkpoint.
dev shared Stage 1 scoring: {'intent_accuracy': 86.57156910969012, 'slot_f1': 80.63007629830174, 'tp': 1638, 'fp': 413, 'fn': 374}
dev original span scoring: {'slot_f1': 80.63007629830174, 'tp': 1638, 'fp': 413, 'fn': 374}
test shared Stage 1 scoring: {'intent_accuracy': 85.07061197041023, 'slot_f1': 78.53127196064652, 'tp': 2235, 'fp': 642, 'fn': 580}
test original span scoring: {'slot_f1': 78.53127196064652, 'tp': 2235, 'fp': 642, 'fn': 580}


## Export results and compare with the new Stage 1 evaluation
Stage 1 values come from `my_stage1_evaluation.json`; its checkpoint hash is verified. Differences are percentage points, not relative percentages. Per-example predictions allow independent rescoring.

In [7]:
stage1 = json.loads(Path('my_stage1_evaluation.json').read_text())
assert hashlib.sha256(Path(stage1['checkpoint']).read_bytes()).hexdigest() == stage1['checkpoint_sha256']
results = pd.DataFrame([{'Model':'Stage-2', 'Split':label, 'Total parameters':total_parameters,
    'Trainable parameters':trainable_parameters, 'Intent accuracy (%)':scores[split]['intent_accuracy'],
    'Slot micro-F1 (%)':scores[split]['slot_f1'], 'Best epoch':checkpoint['epoch']}
    for split,label in [('dev','Validation'),('test','Test')]])
evaluation_record = dict(checkpoint=checkpoint_path,
    checkpoint_sha256=hashlib.sha256(Path(checkpoint_path).read_bytes()).hexdigest(),
    config=config, best_epoch=int(checkpoint['epoch']), total_parameters=total_parameters,
    trainable_parameters=trainable_parameters, validation=scores['dev'], test=scores['test'],
    scoring='shared_evaluation.py: duplicate-aware exact (type, text) micro-F1',
    original_span_scores=original_span_scores,
    provenance={name:hashlib.sha256(Path(name).read_bytes()).hexdigest()
        for name in ['processed.pkl','shared_evaluation.py','my_stage1_evaluation.json']},
    environment=dict(python=platform.python_version(),torch=torch.__version__,device=str(device)),
    saved_validation_metrics=checkpoint['val_metrics'])
Path('my_stage2_evaluation.json').write_text(json.dumps(evaluation_record,indent=2)+'\n')
results.to_csv('my_stage2_results.csv',index=False,float_format='%.2f')
out = Path('evaluation_results'); out.mkdir(exist_ok=True)
for split,records in predictions.items():
    with (out / f'stage2_{split}_predictions.jsonl').open('w',encoding='utf-8') as f:
        for r in records: f.write(json.dumps(r,ensure_ascii=False)+'\n')
comparison=[]
for split,label,key in [('dev','Validation','validation'),('test','Test','test')]:
    comparison.append(dict(Split=label, Stage1_intent=stage1[key]['intent_accuracy'], Stage2_intent=scores[split]['intent_accuracy'],
        Intent_change_pp=scores[split]['intent_accuracy']-stage1[key]['intent_accuracy'],
        Stage1_slot_F1=stage1[key]['slot_f1'],Stage2_slot_F1=scores[split]['slot_f1'],
        Slot_change_pp=scores[split]['slot_f1']-stage1[key]['slot_f1']))
comparison = pd.DataFrame(comparison)
comparison.to_csv(out / 'stage_comparison.csv',index=False,float_format='%.4f')
print(results.to_string(index=False))
print(comparison.to_string(index=False))


  Model      Split  Total parameters  Trainable parameters  Intent accuracy (%)  Slot micro-F1 (%)  Best epoch
Stage-2 Validation           4441930               4441930            86.571569          80.630076          39
Stage-2       Test           4441930               4441930            85.070612          78.531272          39
     Split  Stage1_intent  Stage2_intent  Intent_change_pp  Stage1_slot_F1  Stage2_slot_F1  Slot_change_pp
Validation      86.079685      86.571569          0.491884       79.289657       80.630076        1.340419
      Test      85.272360      85.070612         -0.201748       75.614367       78.531272        2.916905


## Error analysis
Count intent confusions, invalid BIO transitions, and examples with any intent or slot error. Examples are the first errors in dataset order, not hand-picked. Type-level slot counts use the same exact-text multiset comparison as the primary score.

In [8]:
analysis={}
for split,records in predictions.items():
    confusions=Counter(); by_type={}; invalid=0; errors=[]
    for r in records:
        if r['gold_intent'] != r['predicted_intent']:
            confusions[(r['gold_intent'],r['predicted_intent'])]+=1
        previous='O'
        for tag in r['predicted_tags']:
            if tag.startswith('I-') and previous not in ('B-'+tag[2:], 'I-'+tag[2:]): invalid+=1
            previous=tag
        gold=Counter(tuple(s) for s in r['gold_slots']); pred=Counter(tuple(s) for s in r['predicted_slots'])
        for category,counts in [('tp',gold & pred),('fp',pred-gold),('fn',gold-pred)]:
            for (kind,text),count in counts.items():
                by_type.setdefault(kind,dict(tp=0,fp=0,fn=0))[category]+=count
        if gold != pred or r['gold_intent'] != r['predicted_intent']: errors.append(r)
    assert all(sum(v[k] for v in by_type.values()) == scores[split][k] for k in ['tp','fp','fn'])
    analysis[split]=dict(intent_errors=sum(confusions.values()), examples_with_any_error=len(errors),
        invalid_BIO_transitions=invalid, top_intent_confusions=[dict(gold=g,predicted=p,count=c) for (g,p),c in confusions.most_common(10)],
        slots_by_type=by_type, first_five_errors=errors[:5])
    print(split, 'intent errors:',sum(confusions.values()),'| any error:',len(errors),'| invalid BIO:',invalid)
    print('Top confusions:', analysis[split]['top_intent_confusions'][:3])
    for r in errors[:2]:
        print(r['utterance'])
        print('Intent:',r['gold_intent'],'->',r['predicted_intent'])
        print('Slots:',r['gold_slots'],'->',r['predicted_slots'])
(out / 'stage2_error_analysis.json').write_text(json.dumps(analysis,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')


dev intent errors: 273 | any error: 636 | invalid BIO: 3
Top confusions: [{'gold': 'qa_factoid', 'predicted': 'general_quirky', 'count': 9}, {'gold': 'qa_factoid', 'predicted': 'qa_definition', 'count': 7}, {'gold': 'general_quirky', 'predicted': 'news_query', 'count': 6}]
make a room darker
Intent: iot_hue_lightdim -> iot_hue_lightup
Slots: [('house_place', 'room')] -> [('house_place', 'room')]
cleaning is good dust is so bad do now your magic clean my carpet
Intent: iot_cleaning -> general_quirky
Slots: [('house_place', 'carpet')] -> []
test intent errors: 444 | any error: 978 | invalid BIO: 4
Top confusions: [{'gold': 'qa_factoid', 'predicted': 'general_quirky', 'count': 16}, {'gold': 'general_quirky', 'predicted': 'qa_factoid', 'count': 12}, {'gold': 'general_quirky', 'predicted': 'news_query', 'count': 10}]
pink is all we need
Intent: iot_hue_lightchange -> calendar_query
Slots: [('color_type', 'pink')] -> [('color_type', 'pink')]
and the darkness has fallen
Intent: iot_hue_lighto

19107